### ChromaDB

In [39]:
import chromadb
import uuid

#### Chroma Clients
Chroma has several client types, each suited to a different deployment scenario:

**1. EphemeralClient() / Client()**

In-memory only, nothing persists after the process ends. Use for quick tests, experiments, or throwaway scripts.

**2. PersistentClient(path="./chroma_db")**

Local, file-based storage on disk, no server needed. Use when you want Chroma embedded directly in your app (like SQLite), for smaller-scale projects, local development, or when data privacy means you don't want anything on a remote server.

**3. HttpClient(host="localhost", port=8000)**
Connects to a Chroma server you run yourself, self-hosted, Docker, on-prem, your own VM. Use when you need multiple processes/clients to share the same data, or when running Chroma as a proper service.

**4. AsyncHttpClient(...)**

Same as HttpClient but async, for use in async/await codebases (FastAPI, async workers, etc). Same behavior, non-blocking calls.

**5. CloudClient(api_key="ck-...")**

Connects to Chroma Cloud, Chroma's managed hosted service. Handles authentication and endpoint configuration automatically. Use when you don't want to manage your own server infrastructure. Chroma Cookbook

**6. RustClient(path=...)**

Provides access to ChromaDB's Rust-based implementation via Python bindings, offering performance benefits, and is the modern alternative to PersistentClient and EphemeralClient. Worth using if you want better performance than the older Python-native persistent/ephemeral implementations. DeepWiki

**7. AdminClient**

Not for data operations, used for admin tasks like managing tenants/databases (creating/listing them), mainly relevant in multi-tenant self-hosted or cloud setups.

#### Rough decision guide for your use case (terms-of-service RAG project):

- **EphemeralClient / Client:** Quick local prototyping/testing 
- **PersistentClient:** Single-machine app, want persistence 
- **HttpClient:** Multiple services/apps need to hit the same Chroma instance 
- **AsyncHttpClient:** Same as above but in async code 
- **CloudClient:** Don't want to manage a server at all 
- **RustClient:** Want max local performance 

In [140]:
client = chromadb.Client()
client = chromadb.EphemeralClient()

In [ ]:

collection_name = "terms_and_service"
# client.delete_collection(collection_name)

collection = client.get_or_create_collection(name=collection_name)

with open(f"{collection_name}.txt", encoding="utf-8", mode="r") as f:
    terms: list[str] = f.read().splitlines()
    
collection.upsert(
    ids=[ str(uuid.uuid4()) for _ in terms ],
    documents=terms,
    metadatas=[ {"line": line, "source": collection_name } for line in range(len(terms)) ]
) 


In [131]:
collection.peek(limit=2)

{'ids': ['de841dce-450c-4cef-9028-705f90903d7d',
  '706e8177-6f74-48ec-8445-3392d2d7c36e'],
 'embeddings': array([[-1.07899129e-01, -1.62016656e-02, -1.25381513e-03,
         -8.54601562e-02, -4.73183990e-02, -1.33130746e-02,
          1.06241420e-01, -6.21503405e-02,  1.56687405e-02,
          2.74697016e-03,  9.55895812e-04,  3.01126279e-02,
          4.58496287e-02,  4.24250076e-03,  4.12354395e-02,
         -6.74253628e-02,  6.87099919e-02, -3.89264077e-02,
         -1.59801859e-02,  4.08486351e-02,  1.98102724e-02,
          6.98620528e-02, -8.94759297e-02,  6.08924627e-02,
         -1.59394927e-02, -2.97732148e-02, -4.35078330e-02,
          1.62550937e-02,  1.83560830e-02, -3.73599753e-02,
          2.65775286e-02,  7.24719837e-02,  1.01547107e-01,
          2.52541415e-02,  3.29354196e-03, -1.60105191e-02,
         -3.46183144e-02, -4.11830023e-02, -1.63180139e-02,
          4.15207632e-03, -3.96682844e-02, -5.48437759e-02,
         -7.13058487e-02,  2.29976550e-02, -2.32877923

In [ ]:
result = collection.get(
    include=["embeddings", "documents", "metadatas", ]
)

print(len(result['embeddings'][0]))
print(result)

384
{'ids': ['35f818d7-0a62-4dc8-aed3-03624c0cf952', '7a415d36-988c-4fed-9c32-d738bec7d837', '7487a241-7d69-4936-913e-08c84ab5c413', 'ef909b73-9c56-49e9-8935-adcf712f7c87', 'bdd1a287-d2f4-4735-a957-2a8c9d5aeb58', '5202d7c5-3a34-49c0-b7de-3255ec3a3f54', '8aa73a0b-1c37-4143-b920-080619e2a614', '1e99559f-0f76-4df1-b285-232ae7f9f50a', 'd6d93da5-08e5-41f4-b10e-cad22f4a5d0a', '89e09909-891b-4d12-8355-f8e22d335ed5', '3c80d32a-3427-4264-9543-14545cde69a4', '14f8d57f-afb6-4e27-960c-212ce91e454f', 'a291fa9e-390f-4c66-ac54-fd946b689d3f', 'b1d3a31b-c96f-4bf1-a913-2c325a54dbc2', '962edc22-ae4d-42c6-8684-a40ded3a64d8', '3a691732-0c9c-4bb7-a6d0-429e6fdb24b8', 'c38260a5-ae6a-4779-84de-d71e154bf7ba', '5d55d61e-4602-48df-8337-75daf37cdd68', 'fdd9549b-a289-451a-a8ef-e1323e170683', '95140f18-5a72-4975-a9a8-16d680d9fa49', 'b4783e18-9499-4b63-87d1-9c0d923e2966', '6423d3d3-2f4a-4d6f-b9a1-25d8bfc127f6', 'decf5e14-0151-4a1b-ab43-4f88ff68245e', '897f1aa6-dbc6-49df-83c2-d1a5782d34ec', '56a8b7ad-297b-4463-a377-17

In [137]:
results = collection.query(
    query_texts = [
    "Tell me about the Reviews and content",
    "What does the document says about vendors?",
    ],
    include=["documents"],
    n_results=5
)
results

{'ids': [['4110b957-e884-48b5-808b-c9f8496187ae',
   'cdd3bf80-fb05-4f20-9543-c9e114d81f61',
   '52730eca-5809-413f-acef-95e2a23d0a3d',
   '5202d7c5-3a34-49c0-b7de-3255ec3a3f54',
   '7a415d36-988c-4fed-9c32-d738bec7d837'],
  ['b4783e18-9499-4b63-87d1-9c0d923e2966',
   '6423d3d3-2f4a-4d6f-b9a1-25d8bfc127f6',
   '897f1aa6-dbc6-49df-83c2-d1a5782d34ec',
   'b1d3a31b-c96f-4bf1-a913-2c325a54dbc2',
   '5b4ccff5-4ee2-4112-a7f5-ad347d186426']],
 'embeddings': None,
 'documents': [['7. Reviews and content',
   'Post reviews that are false, defamatory, or not based on a genuine order.',
   'You keep ownership of content you submit (such as reviews and photos), but you grant us a non-exclusive licence to display it on the platform. We may remove content that breaches these terms.',
   '1. About us',
   'Last updated: July 13, 2026'],
  ['5. Vendors',
   'Vendors are independent businesses responsible for preparing food safely, describing menu items accurately, honouring their published operating h

In [154]:
collections = client.list_collections()
policies_collection = client.get_collection("policies")

print(f"Collections: {collections}")


Collections: [Collection(name=policies), Collection(name=terms_and_service)]


In [160]:
policies_collection.modify(
   name="policies_modified",
   metadata={"description": "modified policies collection"}
)

collections = client.list_collections()
print(f"Collections: {collections}")
print(f"Document Lenght: {policies_collection.count()}")


Collections: [Collection(name=policies_modified), Collection(name=terms_and_service)]
Document Lenght: 50


In [ ]:
# Without embeddings argument 

dummy_collection = client.get_or_create_collection(name="dummy")
dummy_collection.add(
    ids=["id1", "id2"],
    documents=["love", "hate"],
    metadatas=[{"source": "life"}, {"source": "life"}, ]
)
dummy_collection.peek(1)

{'ids': ['id1'],
 'embeddings': array([[-1.40273377e-01,  2.92286836e-02,  5.19795753e-02,
          1.18946983e-02, -1.75787311e-03,  5.07151559e-02,
          1.12522081e-01,  2.95035001e-02,  1.24193348e-01,
         -6.26634632e-04,  2.63037346e-03, -2.46856939e-02,
         -1.89294294e-02,  4.42330241e-02, -2.61386968e-02,
          1.35184275e-02, -2.94501223e-02, -5.63403703e-02,
         -1.19634360e-01,  3.10926400e-02, -1.12568095e-01,
         -3.20208631e-02, -1.41417999e-02, -1.27476398e-02,
         -4.53569070e-02,  8.78194422e-02,  3.63546908e-02,
         -1.76956840e-02, -2.46265121e-02, -7.72118866e-02,
          2.86627021e-02,  2.12005563e-02,  4.80390601e-02,
          1.57078467e-02, -6.97041601e-02,  6.22597374e-02,
         -3.74162979e-02,  1.42507618e-02,  2.74694636e-02,
         -4.87740897e-02, -1.38656758e-02, -4.24338207e-02,
          2.16306243e-02,  1.10182641e-02, -5.52376211e-02,
          5.93370385e-02,  1.43456254e-02,  3.75540112e-04,
         

### Adding data

In [168]:
client.delete_collection("dummy")
dummy_collection = client.get_or_create_collection(name="dummy")

dummy_collection.add(
    ids=["id1", "id2"],
    # Add embeddings and document
    embeddings=[[0.65, 0.89, 0.4], [-0.08, 0.78, -0.98]],
    documents=["love", "hate"],
    metadatas=[{"source": "life"}, {"source": "life"}, ]
)
dummy_collection.peek(1)

{'ids': ['id1'],
 'embeddings': array([[0.64999998, 0.88999999, 0.40000001]]),
 'documents': ['love'],
 'uris': None,
 'included': ['metadatas', 'documents', 'embeddings'],
 'data': None,
 'metadatas': [{'source': 'life'}]}

In [172]:
dummy_collection.add(
    ids=["id1", "id2", "id3"],
    embeddings=[[1.1, 2.3, 3.2], [4.5, 6.9, 4.4], [1.1, 2.3, 3.2]],
    metadatas=[{"chapter": 3, "verse": 16}, {"chapter": 3, "verse": 5}, {"chapter": 29, "verse": 11}],
)
dummy_collection.peek()

{'ids': ['id1', 'id2', 'id3'],
 'embeddings': array([[ 0.64999998,  0.88999999,  0.40000001],
        [-0.08      ,  0.77999997, -0.98000002],
        [ 1.10000002,  2.29999995,  3.20000005]]),
 'documents': ['love', 'hate', None],
 'uris': None,
 'included': ['metadatas', 'documents', 'embeddings'],
 'data': None,
 'metadatas': [{'source': 'life'},
  {'source': 'life'},
  {'chapter': 29, 'verse': 11}]}

### Updating data

In [179]:
dummy_collection.update(
    ids=["id3"],
    embeddings=[[1.1, 2.3, 3.2]],
    documents=["No longer NONE"],
    metadatas=[ {"chapter": 29, "verse": 11}],
)
dummy_collection.peek()

{'ids': ['id1', 'id2', 'id3'],
 'embeddings': array([[ 0.64999998,  0.88999999,  0.40000001],
        [-0.08      ,  0.77999997, -0.98000002],
        [ 1.10000002,  2.29999995,  3.20000005]]),
 'documents': ['love', 'hate', 'No longer NONE'],
 'uris': None,
 'included': ['metadatas', 'documents', 'embeddings'],
 'data': None,
 'metadatas': [{'source': 'life'},
  {'source': 'life'},
  {'chapter': 29, 'verse': 11}]}

In [180]:
dummy_collection.upsert(
    ids=["id3"],
    embeddings=[[1.1, 2.3, 3.2]],
    documents=["UPSERT No longer NONE"],
    metadatas=[ {"chapter": 29, "verse": 11}],
)
dummy_collection.peek()

{'ids': ['id1', 'id2', 'id3'],
 'embeddings': array([[ 0.64999998,  0.88999999,  0.40000001],
        [-0.08      ,  0.77999997, -0.98000002],
        [ 1.10000002,  2.29999995,  3.20000005]]),
 'documents': ['love', 'hate', 'UPSERT No longer NONE'],
 'uris': None,
 'included': ['metadatas', 'documents', 'embeddings'],
 'data': None,
 'metadatas': [{'source': 'life'},
  {'source': 'life'},
  {'chapter': 29, 'verse': 11}]}

### Delete Data

In [183]:
dummy_collection.delete(ids=["id3"])

{'deleted': 1}

In [182]:
dummy_collection.delete(where={"chapter": 29})

{'deleted': 0}

In [187]:
dummy_collection.query(
    query_embeddings=[
        [1,0.7,0.6],
        [0.4,0.7,-0.4]
    ],
)

{'ids': [['id1', 'id2'], ['id2', 'id1']],
 'embeddings': None,
 'documents': [['love', 'hate'], ['hate', 'love']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'source': 'life'}, {'source': 'life'}],
  [{'source': 'life'}, {'source': 'life'}]],
 'distances': [[0.19860002398490906, 3.6692001819610596],
  [0.5732001066207886, 0.7386000156402588]]}